# Six estimators, one number, and the assumptions each needs

Difference-in-differences, synthetic control, a time-based regression, a cluster-based one, a
switchback, a ghost-bid comparison. Every one of them returns "the effect". They are not
interchangeable — each is a causal estimate only under assumptions the others do not need, and
running the wrong one on a design it was not built for returns a number that looks exactly
like the right one.

Six estimators share one data contract: a `PanelArrays` — an `(n_units, n_periods)` outcome
array, the treated row indices, half-open `pre`/`post` windows, and where needed a switchback
`assignment` or a ghost `exposed` mask — and one output, a `MethodEstimate` whose `effect` is
always the *average effect per treated unit per post period* with a Wald interval at the
requested mass, or a typed `Unsupported`. Each `MethodSpec` names the `core.Assumption`s
under which its number is a causal effect.

In [ ]:
import numpy as np

from axiom.core import Unsupported
from axiom.design import (
    ASSUMPTIONS, METHODS, MethodEstimate, MethodName, MethodSpec, MethodStatus, PanelArrays,
    SimulationSpec, bartlett_bandwidth, estimate, estimate_cluster_based_regression,
    estimate_difference_in_differences, estimate_ghost, estimate_switchback,
    estimate_synthetic_control, estimate_time_based_regression, method_assumption, method_spec,
    panel_arrays, simplex_weights, simulate_panel,
)

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, compare, intervals

enable();  # every axiom result renders itself from here on

In [ ]:
table(
    [
        [name, spec.status, str(spec.requires_pre_period), str(spec.requires_controls),
         str(spec.assumption_names())]
        for name, spec in METHODS.items()
    ],
    headers=("method", "status", "needs pre-period", "needs controls", "assumptions"),
)
print()
print("assumption 'parallel_trends':", ASSUMPTIONS["parallel_trends"].statement)
extra = method_assumption("balanced_exposure", "exposure is balanced across arms", "an imbalance test")
print("custom assumption:", extra.facet, extra.name, "| challenged by:", extra.challenged_by)

In [ ]:
ms: MethodSpec = method_spec("switchback")
status: MethodStatus = ms.status
name: MethodName = "switchback"
print(name, status, ms.description)

## A seeded synthetic panel

`simulate_panel` draws `y[i, t] = mu_i + s_t + e[i, t] + effect · D[i, t]` once; the panel
carries the treated set, the windows, a switchback `assignment` and an `exposed` mask, so every
registered method can read it.

In [ ]:
spec = SimulationSpec(n_units=16, n_periods=20, n_pre=10, n_treated=8, effect=1.0, noise_sd=1.0, seed=7)
rng = np.random.default_rng(spec.seed)
arrays: PanelArrays = simulate_panel(spec, rng, design="holdout")
print(arrays.outcome.shape, "treated:", arrays.treated, "pre:", arrays.pre, "post:", arrays.post)

In [ ]:
rows = []
for method in METHODS:
    out = estimate(method, arrays, mass=0.95)
    if isinstance(out, Unsupported):
        rows.append([method, "Unsupported", out.reason, ""])
        continue
    assert isinstance(out, MethodEstimate)
    rows.append(
        [method, f"{out.effect:.3f}", f"[{out.interval.lower:.3f}, {out.interval.upper:.3f}]",
         out.se_method]
    )
table(rows, headers=("method", "effect", "interval", "se method"))

In [ ]:
estimated = []
for method in METHODS:
    out = estimate(method, arrays, mass=0.95)
    if isinstance(out, MethodEstimate):
        estimated.append((method, out.effect, out.interval.lower, out.interval.upper))

fig = intervals(
    estimated, ref=spec.effect if False else 1.0, ref_label="the true effect",
    highlight="switchback",
    title="Six estimators, one holdout panel",
    subtitle="each method's estimate with its 95% Wald interval — the true effect is 1.0",
    x_title="estimated effect per treated unit per post period",
)
caption(fig, "Four estimates around the truth at very different widths, and one — the "
             "switchback — reading an assignment that this panel does not have, returning "
             "approximately zero rather than a plausible effect. That is the failure mode "
             "worth designing for: the wrong method on the wrong design returns a number, and "
             "the number is not marked.")

The four holdout methods recover the true effect of 1.0, at intervals that differ in width by
a factor of two — the choice of method is a precision choice as well as an assumption choice.
The switchback estimator reads the `assignment` array, which on a holdout panel is unrelated
to who was treated, so it correctly returns approximately zero. The ghost estimator reads the
`exposed` mask, which on this panel does track the treatment, so it recovers the effect with a
wide interval. The keyword conveniences below build the `PanelArrays` for you.

In [ ]:
y, tr, pre, post = arrays.outcome, arrays.treated, arrays.pre, arrays.post
did = estimate_difference_in_differences(y, tr, pre, post)
sc = estimate_synthetic_control(y, tr, pre, post, rmspe_ratio_max=5.0)
tbr = estimate_time_based_regression(y, tr, pre, post)
cbr = estimate_cluster_based_regression(y, tr, pre, post)
rows = []
for e in (did, sc, tbr, cbr):
    assert isinstance(e, MethodEstimate)
    rows.append([e.method, f"{e.effect:.3f}", f"{e.se:.3f}", str(sorted(e.detail)[:4])])
table(rows, headers=("method", "effect", "se", "detail keys"))

In [ ]:
# Switchback and ghost panels where the effect actually follows their own designs.
sb_arrays = simulate_panel(spec, np.random.default_rng(1), design="switchback")
sb = estimate_switchback(sb_arrays.outcome, sb_arrays.assignment)
assert isinstance(sb, MethodEstimate)
print(f"switchback: {sb.effect:.3f} ± {sb.se:.3f}  bandwidth={sb.detail.get('bandwidth')}  (rule: {bartlett_bandwidth(spec.n_periods)})")
gh_arrays = simulate_panel(spec, np.random.default_rng(2), design="ghost")
gh = estimate_ghost(gh_arrays.outcome, gh_arrays.treated, gh_arrays.post, gh_arrays.exposed)
assert isinstance(gh, MethodEstimate)
print(f"ghost:      {gh.effect:.3f} ± {gh.se:.3f}  n_treated={gh.n_treated} n_control={gh.n_control}")

## Inside synthetic control

`simplex_weights` solves for non-negative donor weights summing to one that best reproduce the
treated unit's pre-period series.

In [ ]:
donors = y[list(arrays.control)][:, pre]
target = y[tr[0], pre]
w = simplex_weights(target, donors)
print("weights:", np.round(w, 3), "| sum =", round(float(w.sum()), 6), "| min =", round(float(w.min()), 6))
print("pre-period RMSPE:", round(float(np.sqrt(np.mean((target - w @ donors) ** 2))), 4))

In [ ]:
labels = [f"donor {i}" for i in arrays.control]
fig = compare(
    labels, w,
    highlight=labels[int(np.argmax(w))],
    value_fmt="{:.3f}",
    title="What a synthetic control actually is",
    subtitle="non-negative weights summing to one, chosen to reproduce the treated unit's pre-period",
    x_title="weight",
)
caption(fig, "Most donors get nothing. The constraint that the weights are non-negative and "
             "sum to one is what stops the fit from extrapolating — a synthetic control is a "
             "weighted average of real units, never a linear combination that leaves their range.")

## Typed failures, never wrong numbers

A method whose data requirements are not met returns `Unsupported` naming the requirement:
here every unit is treated, so there are no controls.

In [ ]:
all_treated = panel_arrays(y, treated=range(y.shape[0]), pre=pre, post=post)
out = estimate("difference_in_differences", all_treated)
assert isinstance(out, Unsupported)
print(out.reason, "| missing:", out.missing)
no_pre = panel_arrays(y, tr, pre=slice(0, 0), post=slice(0, 20))
print(estimate("synthetic_control", no_pre).reason)

## Small-sample critical values

Every method's SE is an estimated variance with finite degrees of freedom, so the interval uses a Student-t critical value at the recorded `df` (`MethodEstimate.critical == "student_t"`). `wald_t` builds that interval; it is still labelled `wald` because that is what it is: estimate ± critical × se.

In [ ]:
from axiom.design import t_critical, wald_t

table(
    [[df, round(t_critical(0.95, df), 4)] for df in (5, 10, 30, 1000)],
    headers=("df", "t critical at 95%"),
)
print(wald_t(1.0, 0.5, 0.95, df=10))

## What this bought you

Six methods behind one call, each carrying the assumptions its number depends on, each
returning a typed refusal instead of an estimate when its data requirements are not met — and
one interval convention, with the small-sample critical value, across all of them.

`04-simulation.ipynb` puts each of them through an A/A calibration, which is where a method
that quietly does not control its false-positive rate gets caught.